In [21]:
# Subimos el artículo en Word
from google.colab import files
import shutil
from datetime import datetime  # Añade esto al inicio con los otros imports


uploaded = files.upload()
nombre_subido = list(uploaded.keys())[0]
shutil.move(nombre_subido, "articulo.docx")  # Renombra el archivo

# --- Bibliotecas ---
!pip install python-docx
from lxml import etree as ET
from docx import Document

# --- Diccionarios  ---

# Datos de la revista (información que no cambia entre artículos)
revista = {
    "journal_id": "ARIS",
    "journal_title": "Arte, individuo y sociedad",
    "abbrev_journal_title": "Art., indv. y soc.",
    "issn": "1131-5598",
    "issn_format": "electronic",
    "issn_l": "1131-5598",
    "volume": "37",
    "issue": "1",
    "fpage": "181",  # Primera página del artículo
    "lpage": "192",  # Última página del artículo
    "publisher_name": "Universidad Complutense de Madrid",
    "publisher_country": "España",
    "website": "https://revistas.ucm.es/aris"
}

# Namespaces
NSMAP = {
    'xlink': "http://www.w3.org/1999/xlink",
    'mml': "http://www.w3.org/1998/Math/MathML",
    'xsi': "http://www.w3.org/2001/XMLSchema-instance",
    'ali': "http://www.niso.org/schemas/ali/1.0/",
    'oasis': "http://www.niso.org/standards/z39-96/ns/oasis-exchange/table"
}

# --- Función nueva para extraer datos del Word ---

# Diccionario semiautomático

def extraer_de_word(ruta_word):
    doc = Document(ruta_word)
    datos = {
        "tipo_articulo": "research-article",
        "lang": "en",
        "contributors": [],
        # Campos que DEBEN extraerse del Word:
        "article-id": None,       # Estilo: "article-id"
        "article-title": None,    # Estilo: "article-title"
        "trans-title": None,      # Estilo: "trans-title"
        "abstract": None,         # Estilo: "abstract" (resumen español)
        "trans-abstract": None,   # Estilo: "trans-abstract" (abstract inglés)
        "kwd-group": [],          # Estilo: "kwd" (palabras clave español)
        "kwd-group-trans": [],    # Estilo: "kwd-trans" (keywords inglés)
        "subject_section": None,  # Valor por defecto
        "publication_date": {"day": "01", "month": "01", "year": "2024"},
        # Fechas por defecto (se pueden sobrescribir)
        "received_date": {"day": "01", "month": "01", "year": "2023"},
        "revised_date": {"day": "01", "month": "01", "year": "2024"},
        "accepted_date": {"day": "01", "month": "01", "year": "2024"},

    }

# Función que extrae la información según los estilos

    for p in doc.paragraphs:
        if not p.style.name:
            continue

        if p.style.name == "article-title":
            datos["article-title"] = p.text
        elif p.style.name == "article-id":
            datos["article-id"] = p.text
        elif p.style.name == "trans-title":
            datos["trans-title"] = p.text
        elif p.style.name == "abstract":
            datos["abstract"] = p.text
        elif p.style.name == "trans-abstract":
            datos["trans-abstract"] = p.text
        elif p.style.name == "kwd":
            datos["kwd-group"].append(p.text.strip())
        elif p.style.name == "kwd-trans":
            datos["kwd-group-trans"].append(p.text.strip())
        elif p.style.name == "subject":
            datos["subject_section"] = p.text.strip()
            if datos["subject_section"] is None:
                datos["subject_section"] = "Artículo de investigación"
        elif p.style.name == "pub-date":
            try:
                fecha = datetime.strptime(p.text.strip(), "%Y/%m/%d")
                datos["publication_date"] = {
                    "day": f"{fecha.day:02d}",
                    "month": f"{fecha.month:02d}",
                    "year": str(fecha.year)
                }
            except ValueError:
                print(f"⚠️ Formato de fecha inválido en 'pub-date'. Usando valores por defecto.")
        elif p.style.name == "contrib-group":
            partes = p.text.split("|")
            if len(partes) >= 4:
                datos["contributors"].append({
                    "surname": partes[0].split(",")[0].strip(),
                    "given-names": partes[0].split(",")[1].strip(),
                    "contrib-id": partes[1].strip(),
                    "institution": partes[2].strip(),
                    "country": "España",
                    "country-code": "ES",
                    "email": partes[3].strip(),
                    "aff-id": f"aff-{len(datos['contributors']) + 1}",
                    "corresp-id": f"corresp-{len(datos['contributors']) + 1}"
                })

    return datos


# Extraer datos del artículo
articulo = extraer_de_word("articulo.docx")

# Permisos y licencia
permissions = {
    "copyright_statement": "Copyright © 2024, Universidad Complutense de Madrid",
    "copyright_year": "2024",
    "copyright_holder": "Universidad Complutense de Madrid",
    "license_href": "https://creativecommons.org/licenses/by/4.0/",
    "license_ref": "https://creativecommons.org/licenses/by/4.0/",
    "license_p_text": "Esta obra está bajo una licencia ",
    "license_p_link_text": "Creative Commons Attribution 4.0 International"
}

# --- Generación del XML JATS  ---

# 1. Elemento raíz
article = ET.Element(
    "article",
    **{
        "article-type": articulo["tipo_articulo"],
        "dtd-version": "1.3"
    },
    nsmap=NSMAP
)
article.set("{http://www.w3.org/XML/1998/namespace}lang", articulo["lang"])

# 2. Front
front = ET.SubElement(article, "front")

# 3. Journal Meta
journal_meta = ET.SubElement(front, "journal-meta")
ET.SubElement(journal_meta, "journal-id", **{"journal-id-type": "publisher-id"}).text = revista["journal_id"]

title_group = ET.SubElement(journal_meta, "journal-title-group")
ET.SubElement(title_group, "journal-title", **{"specific-use": "original"}).text = revista["journal_title"]
ET.SubElement(title_group, "abbrev-journal-title", **{"abbrev-type": "publisher"}).text = revista["abbrev_journal_title"]

ET.SubElement(journal_meta, "issn", **{"publication-format": revista["issn_format"]}).text = revista["issn"]
ET.SubElement(journal_meta, "issn-l").text = revista["issn_l"]

publisher = ET.SubElement(journal_meta, "publisher")
ET.SubElement(publisher, "publisher-name").text = revista["publisher_name"]
ET.SubElement(ET.SubElement(publisher, "publisher-loc"), "country").text = revista["publisher_country"]

# 4. Article Meta
article_meta = ET.SubElement(front, "article-meta")

# 5. Article ID
ET.SubElement(article_meta, "article-id", **{"pub-id-type": "doi"}).text = articulo["article-id"]

# 6. Article Categories
subj_group = ET.SubElement(ET.SubElement(article_meta, "article-categories"), "subj-group", **{"subj-group-type": "section"})
ET.SubElement(subj_group, "subject").text = articulo["subject_section"]

# 7. Title Group
title_group = ET.SubElement(article_meta, "title-group")
ET.SubElement(title_group, "article-title").text = articulo["article-title"]

# 7.1. Título traducido (con xml:lang según el idioma contrario)
trans_lang = "es" if articulo["lang"] == "en" else "en"
trans_title_group = ET.SubElement(title_group, "trans-title-group", **{"{http://www.w3.org/XML/1998/namespace}lang": trans_lang})
ET.SubElement(trans_title_group, "trans-title").text = articulo["trans-title"]

# 8. Contrib Group (autores)
contrib_group = ET.SubElement(article_meta, "contrib-group")
for c in articulo["contributors"]:
    contrib = ET.SubElement(contrib_group, "contrib", **{"contrib-type": "author", "corresp": "yes" if "corresp-id" in c else "no"})
    ET.SubElement(contrib, "contrib-id", **{"contrib-id-type": "orcid"}).text = c["contrib-id"]
    name = ET.SubElement(contrib, "name")
    ET.SubElement(name, "surname").text = c["surname"]
    ET.SubElement(name, "given-names").text = c["given-names"]
    ET.SubElement(contrib, "xref", **{"ref-type": "aff", "rid": c["aff-id"]})
    if "corresp-id" in c:
        ET.SubElement(contrib, "xref", **{"ref-type": "corresp", "rid": c["corresp-id"]})

# 8.1. Afiliaciones
for c in articulo["contributors"]:
    aff = ET.SubElement(contrib_group, "aff", **{"id": c["aff-id"]})
    ET.SubElement(aff, "institution", **{"content-type": "original"}).text = c["institution"]
    country_elem = ET.SubElement(aff, "country", country=c["country-code"])
    country_elem.text = c["country"]

# 8.2. Author Notes
author_notes = ET.SubElement(article_meta, "author-notes")
for c in [x for x in articulo["contributors"] if "corresp-id" in x]:
    corresp = ET.SubElement(author_notes, "corresp", **{"id": c["corresp-id"]})
    corresp.text = f"{c['given-names']} {c['surname']}"
    ET.SubElement(corresp, "email").text = c["email"]

# 9. Fecha publicación
pub_date = ET.SubElement(article_meta, "pub-date",
                       **{"date-type": "pub",
                          "publication-format": "electronic",
                          "iso-8601-date": f"{articulo['publication_date']['year']}-{articulo['publication_date']['month']}-{articulo['publication_date']['day']}"})
ET.SubElement(pub_date, "day").text = articulo["publication_date"]["day"]
ET.SubElement(pub_date, "month").text = articulo["publication_date"]["month"]
ET.SubElement(pub_date, "year").text = articulo["publication_date"]["year"]

# 10. Volumen de la revista, número y páginas

# 10.1. Volumen y número
ET.SubElement(article_meta, "volume").text = revista["volume"]
ET.SubElement(article_meta, "issue").text = revista["issue"]

# 10.2. Paginación (formato JATS 1.3 válido)
ET.SubElement(article_meta, "fpage").text = revista["fpage"]
ET.SubElement(article_meta, "lpage").text = revista["lpage"]

# 11. History (recibido, revisado y aceptado)
history = ET.SubElement(article_meta, "history")

# 11.1. Fecha recibido
date_data = articulo["received_date"]
date_elem = ET.SubElement(history, "date",
                        **{"date-type": "received",
                           "iso-8601-date": f"{date_data['year']}-{date_data['month']}-{date_data['day']}"})
ET.SubElement(date_elem, "day").text = date_data["day"]
ET.SubElement(date_elem, "month").text = date_data["month"]
ET.SubElement(date_elem, "year").text = date_data["year"]

# 11.2. Fecha revisión
date_data = articulo["revised_date"]
date_elem = ET.SubElement(history, "date",
                        **{"date-type": "rev-recd",
                           "iso-8601-date": f"{date_data['year']}-{date_data['month']}-{date_data['day']}"})
ET.SubElement(date_elem, "day").text = date_data["day"]
ET.SubElement(date_elem, "month").text = date_data["month"]
ET.SubElement(date_elem, "year").text = date_data["year"]

# 11.3. Fecha aceptación
date_data = articulo["accepted_date"]
date_elem = ET.SubElement(history, "date",
                        **{"date-type": "accepted",
                           "iso-8601-date": f"{date_data['year']}-{date_data['month']}-{date_data['day']}"})
ET.SubElement(date_elem, "day").text = date_data["day"]
ET.SubElement(date_elem, "month").text = date_data["month"]
ET.SubElement(date_elem, "year").text = date_data["year"]

# 12. Permissions
perms_elem = ET.SubElement(article_meta, "permissions")
ET.SubElement(perms_elem, "copyright-statement").text = permissions["copyright_statement"]
ET.SubElement(perms_elem, "copyright-year").text = permissions["copyright_year"]
ET.SubElement(perms_elem, "copyright-holder").text = permissions["copyright_holder"]

license_elem = ET.SubElement(perms_elem, "license",
                           **{"license-type": "open-access",
                              "{http://www.w3.org/1999/xlink}href": permissions["license_href"]})
ET.SubElement(license_elem, "{http://www.niso.org/schemas/ali/1.0/}license_ref").text = permissions["license_ref"]

license_p = ET.SubElement(license_elem, "license-p")
license_p.text = permissions["license_p_text"]
ET.SubElement(license_p, "ext-link",
             **{"ext-link-type": "uri",
                "{http://www.w3.org/1999/xlink}href": permissions["license_href"]}).text = permissions["license_p_link_text"]

# 13. Abstract y Trans-Abstract
abstract_elem = ET.SubElement(article_meta, "abstract")
ET.SubElement(abstract_elem, "p").text = articulo["abstract"]

# 13.1. Abstract traducido (con xml:lang contrario)
trans_abstract_lang = "es" if articulo["lang"] == "en" else "en"
trans_abstract = ET.SubElement(article_meta, "trans-abstract", **{"{http://www.w3.org/XML/1998/namespace}lang": trans_abstract_lang})
ET.SubElement(trans_abstract, "p").text = articulo["trans-abstract"]

# 14. Keywords
kwd_group_main = ET.SubElement(article_meta, "kwd-group", **{"kwd-group-type": "author-keywords"})
for kwd in articulo["kwd-group"]:
    ET.SubElement(kwd_group_main, "kwd").text = kwd

# 14.1. Keywords traducidas (con xml:lang contrario)
trans_kwd_lang = "es" if articulo["lang"] == "en" else "en"
kwd_group_trans = ET.SubElement(article_meta, "kwd-group", **{
    "kwd-group-type": "author-keywords",
    "{http://www.w3.org/XML/1998/namespace}lang": trans_kwd_lang
})
for kwd in articulo["kwd-group-trans"]:
    ET.SubElement(kwd_group_trans, "kwd").text = kwd

# --- Resto del código (body, back, generación XML) se mantiene igual ---

# 15. Body

body = ET.SubElement(article, "body")
sec = ET.SubElement(body, "sec")
title = ET.SubElement(sec, "title")
title.text = "Segunda fase de la edición:"
p = ET.SubElement(sec, "p")
p.text = "Añadir cuerpo del artículo con Pandoc."

# 16. Back

back = ET.SubElement(article, "back")
ref_list = ET.SubElement(back, "ref-list")
ref1 = ET.SubElement(ref_list, "ref", id="ref1")
element_citation1 = ET.SubElement(ref1, "element-citation", {"publication-type": "journal"})
ET.SubElement(element_citation1, "person-group", {"person-group-type": "author"}).text = "Apellido1, Nombre1"
ET.SubElement(element_citation1, "year").text = "2023"
ET.SubElement(element_citation1, "article-title").text = "Título del artículo de ejemplo"
ET.SubElement(element_citation1, "source").text = "Nombre de la revista"
ET.SubElement(element_citation1, "volume").text = "15"
ET.SubElement(element_citation1, "fpage").text = "123"
ET.SubElement(element_citation1, "lpage").text = "145"
ET.SubElement(element_citation1, "pub-id", {"pub-id-type": "doi"}).text = "10.1234/ejemplo.doi"

# 17. Declaraciones iniciales del documento XML que vamos a general

DOCTYPE = '''<!DOCTYPE article PUBLIC "-//NLM//DTD JATS (Z39.96) Journal Publishing DTD with OASIS Tables with MathML3 v1.3 20210610//EN" "https://jats.nlm.nih.gov/publishing/1.3/JATS-journalpublishing-oasis-article1-3-mathml3.dtd">'''
XML_MODEL = '<?xml-model type="application/xml-dtd" href="http://jats.nlm.nih.gov/publishing/1.3/JATS-journalpublishing-oasis-article1-3-mathml3.dtd"?>'

xml_str = ET.tostring(
    article,
    encoding='UTF-8',
    pretty_print=True,
    xml_declaration=False
)

with open('articulo_jats.xml', 'wb') as f:
    f.write(b'<?xml version="1.0" encoding="UTF-8"?>\n')
    f.write(XML_MODEL.encode('UTF-8') + b'\n')
    f.write(DOCTYPE.encode('UTF-8') + b'\n')
    f.write(xml_str)

# 18. Print para comprobar la descarga y enlace al validador

from IPython.display import display, HTML

display(HTML("""
<p>✅ Archivo <strong>articulo_jats.xml</strong> generado correctamente.</p>
<p>Valida tu XML JATS en: <a href="https://jats4r-validator.niso.org/" target="_blank">https://jats4r.org/validator</a></p>
"""))

KeyboardInterrupt: 